# 01 — Current baseline

**Goal:** reproduce the current logistic / XGBoost / SVM setup and pick the best held-out ROC-AUC.

## Why this experiment?
We need a fair starting point. Every later notebook is compared against this one.
If a new idea does not beat this baseline, it is not useful.

## Approach
1. Use the same cleaned data and the shared train/test split.
2. Add a geographic cluster feature with KMeans (3 clusters), fitted on **training rows only**.
3. Train three models: Logistic Regression, XGBoost, SVM.
4. Keep the model with the highest held-out ROC-AUC and save that score.

## What changed vs the original notebook?
- Same models, but evaluation is standardized (ROC-AUC, F1, AP, timing).
- KMeans is fit only on train data (no leakage from the test set).
- Result is written to `results/` for the final leaderboard.

## Features used in this notebook
- **Base raw columns:** category, weekday, time_bucket, distance, products, lat/lon, first fare, final fares
- **Extra engineered feature:** `geo_cluster` from KMeans(3) on pickup+drop-off coordinates
- **How selected:** keep the original project setup; only add a simple geo region label
- **Why:** establishes the floor score before new feature engineering


### Setup
Load shared helpers, data, and the fixed 80/20 split.


In [1]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Features and models
Build baseline features, add train-only geo clusters, define LR / XGB / SVM.


In [2]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

X_train, y_train = make_xy(train_df)
X_test, y_test = make_xy(test_df)

# Fit geographic clustering on training rows only.
geo_cols = ["source_latitude", "source_longitude", "destination_latitude", "destination_longitude"]
geo_imputer = SimpleImputer(strategy="median")
geo_train = geo_imputer.fit_transform(train_df[geo_cols])
geo_test = geo_imputer.transform(test_df[geo_cols])
geo_scaler = StandardScaler()
kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
X_train["geo_cluster"] = kmeans.fit_predict(geo_scaler.fit_transform(geo_train))
X_test["geo_cluster"] = kmeans.predict(geo_scaler.transform(geo_test))

models = {
    "logistic_regression": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    "xgboost": Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1))]),
    "svm_rbf": Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", SVC(C=3.0, probability=True, random_state=RANDOM_STATE))]),
}


### Features selected and why

This experiment uses the **original raw numeric columns** plus one engineered geo cluster.

| Group | Features | Why we keep them |
|-------|----------|------------------|
| Trip basics | distance, products, category | Core job description |
| Time (raw) | weekday, time_bucket | Simple schedule signal |
| Location (raw) | source/dest lat/lon | Area effects |
| Price (raw) | first + final fares | Price attractiveness (final fares may leak) |
| Engineered | geo_cluster | Compresses location into 3 regions, fit on **train only** |

Run the next cell to list every column actually used in `X_train`.


In [ ]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}
cols = list(X_train.columns)
rows = [{"feature": c, "why_selected": feature_why.get(c, "Included from the baseline feature set")} for c in cols]
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Selection rule: original raw columns + train-only KMeans geo_cluster")
feature_table


### Compare models
Score all three on the same held-out test set.


In [3]:
all_metrics = {
    name: evaluate(model, X_train, y_train, X_test, y_test)
    for name, model in models.items()
}
comparison = pd.DataFrame(all_metrics).T.sort_values("roc_auc", ascending=False)
comparison[["roc_auc", "avg_precision", "f1", "accuracy", "fit_seconds"]]


,roc_auc,avg_precision,f1,accuracy,fit_seconds
xgboost,0.973043,0.972153,0.911504,0.918992,0.674
svm_rbf,0.940013,0.936843,0.894447,0.905041,3.324
logistic_regression,0.925506,0.924341,0.871639,0.886139,0.035


### Save best result
Write the winner into `results/` as experiment `01`.


In [4]:
best_name = comparison.index[0]
metrics = all_metrics[best_name]
result_path = save_result(
    "01", "baseline_current", "Current feature set + train-only KMeans(3); select best LR/XGB/SVM",
    metrics, best_model=best_name, notes="The model comparison is performed on the common held-out split.",
    feature_count=X_train.shape[1],
)
result_path


PosixPath('/Users/shahriar/Desktop/Desktop/Work/MyStartUp/R&D/hyperack_exp/results/01_baseline_current.json')

### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report, evaluate

# Refit the winning model so we can inspect actual vs predicted on the test set.
best_model = models[best_name]
best_metrics = evaluate(best_model, X_train, y_train, X_test, y_test)
metrics = {**metrics, **{k: best_metrics[k] for k in ("y_true", "y_pred", "y_prob", "precision") if k in best_metrics}}

avp = actual_vs_predicted_report(
    best_metrics["y_true"],
    best_metrics["y_pred"],
    best_metrics["y_prob"],
    sample_size=25,
)
print(f"Winner model: {best_name}")
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


## What to look at
- Which of LR / XGB / SVM wins?
- This ROC-AUC is the floor for experiments 02–15.
